# Ropedia Academy — B6 · Making NeRF fast: hash grids

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B6.ipynb)

> **Builds the multi-resolution hash encoding (local cell lookup + spatial hash) feeding a tiny MLP, and shows gradients flow into the grid — why Instant-NGP is fast.**
>
> 搭出多分辨率哈希编码（局部单元查找 + 空间哈希）喂给极小 MLP，并展示梯度流入网格——这正是 Instant-NGP 之所以快。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B6

In [ ]:
import torch, torch.nn as nn

# Instant-NGP: put capacity in a MULTI-RESOLUTION HASH GRID; read it with a tiny MLP.
L, Tsize, Fdim = 4, 2**14, 2                         # levels, table size, feature dim
resolutions = [4, 8, 16, 32]
tables = nn.ParameterList([nn.Parameter(torch.randn(Tsize, Fdim) * 1e-2) for _ in range(L)])
primes = torch.tensor([1, 2654435761, 805459861])

def hash_encode(x):                                  # x in [0,1]^3 -> features
    feats = []
    for table, res in zip(tables, resolutions):
        cell = (x * res).long()                       # local grid cell (cheap, local)
        idx  = (cell * primes).sum(-1) % Tsize        # spatial hash (collisions tolerated)
        feats.append(table[idx])
    return torch.cat(feats, -1)                        # (..., L*Fdim)

tiny_mlp = nn.Sequential(nn.Linear(L*Fdim, 64), nn.ReLU(), nn.Linear(64, 4))
x = torch.rand(1000, 3)
out = tiny_mlp(hash_encode(x))
print("encoded", x.shape[0], "points ->", tuple(out.shape), "via a tiny MLP")
out.sum().backward()                                  # gradients flow into the GRID features
print("capacity lives in the grid (trainable):", tables[0].grad is not None, "-> trains in seconds")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B6
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks